# Comparación de algoritmos de ordenamiento · laboratorio ejecutable

Este notebook conserva el código, los controles y la animación. La explicación, las ecuaciones y la interpretación de resultados se encuentran en el complemento digital:

<div style="margin:1.25rem 0 1.5rem;padding:18px 20px;border:1px solid #bdb9b2;border-left:4px solid #8b4b32;background:#fbfaf7;color:#242321;font-family:Arial,Helvetica,sans-serif;">
  <div style="margin-bottom:6px;color:#8b4b32;font-size:11px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;">Complemento digital</div>
  <div style="margin-bottom:14px;font-size:15px;line-height:1.55;">Consulta la explicación, las ecuaciones y la interpretación de resultados en el sitio de la obra.</div>
  <a class="notebook-pages-button" href="https://notas-a-mano-serie-de-libros.github.io/3_notas-a-mano-sobre-analisis-de-complejidad-computacional/capitulos/capitulo-8/0-comparacion-ordenamientos/" target="_blank" rel="noopener noreferrer" aria-label="Leer la explicación completa en GitHub Pages; abre una pestaña nueva">
    <img src="https://img.shields.io/badge/LEER_LA_EXPLICACI%C3%93N_COMPLETA_EN_GITHUB_PAGES-33312e?style=for-the-badge&logo=github&logoColor=white" alt="Leer la explicación completa en GitHub Pages" width="430" />
  </a>
</div>

Ejecuta las celdas en orden y utiliza los controles de la simulación. Al finalizar, vuelve a Pages para contrastar los resultados con el análisis teórico.


**Uso.** Ejecute la celda de simulación. Defina el tamaño del arreglo. Seleccione el orden: ascendente o descendente.


In [ ]:
#@title Ejecutar comparación de ordenamientos { display-mode: "form" }

import time
import urllib.error
import urllib.request
from pathlib import Path

SIMULATION_NAME = "comparacion"
RAW_BOOTSTRAP_URL = "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/3_notas-a-mano-sobre-analisis-de-complejidad-computacional/main/simulaciones/capitulo8/runtime/colab_bootstrap.py"


def find_local_bootstrap():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "simulaciones" / "capitulo8" / "runtime" / "colab_bootstrap.py"
        if candidate.is_file():
            return candidate
    return None


def read_remote_bootstrap():
    for attempt in range(3):
        try:
            with urllib.request.urlopen(RAW_BOOTSTRAP_URL, timeout=30) as response:
                return response.read().decode("utf-8")
        except urllib.error.HTTPError as error:
            if error.code not in {429, 500, 502, 503, 504} or attempt == 2:
                raise
        except urllib.error.URLError:
            if attempt == 2:
                raise
        time.sleep(2 ** attempt)


bootstrap = find_local_bootstrap()
if bootstrap is not None:
    exec(bootstrap.read_text(encoding="utf-8"))
else:
    exec(read_remote_bootstrap())


**Eficiencia por tamaño de arreglo.** Ejecute esta celda para comparar las operaciones al aumentar el tamaño del arreglo. Active la curva teórica para contrastarla con las mediciones; la línea discontinua muestra la extrapolación. Seleccione los algoritmos que desea mostrar. La medición puede tardar varios minutos.


In [ ]:
#@title Eficiencia por tamaño de arreglo { display-mode: "form" }
import urllib.request
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

RAW_BASE_URL = "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/3_notas-a-mano-sobre-analisis-de-complejidad-computacional/main/core/sort"
CHART_CANDIDATES = (
    *(base / "core/sort/ordenamientos_chart.py" for base in (Path.cwd(), *Path.cwd().parents)),
    Path("core/sort/ordenamientos_chart.py"),
    Path("ordenamientos_chart.py"),
)
chart_path = next((path for path in CHART_CANDIDATES if path.exists()), None)
if chart_path is None:
    chart_path = Path("ordenamientos_chart.py")
    chart_path.write_text(urllib.request.urlopen(f"{RAW_BASE_URL}/ordenamientos_chart.py").read().decode("utf-8"), encoding="utf-8")
    Path("sort_metrics.py").write_text(urllib.request.urlopen(f"{RAW_BASE_URL}/sort_metrics.py").read().decode("utf-8"), encoding="utf-8")

spec = spec_from_file_location("cap8_ordenamientos_chart_runtime", chart_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f"No se pudo cargar {chart_path}")

chart_module = module_from_spec(spec)
spec.loader.exec_module(chart_module)
chart_module.run_chart()
